# RAG Pipeline: Cytiva Pharmaceutical Document Analysis
Author: Cheryl  
Tools: LlamaIndex + Google Gemini  

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline that:
1. Parses a multi-page Cytiva PDF (storage letter + certificates of quality)
2. Chunks the text into meaningful segments
3. Generates vector embeddings with Gemini
4. Retrieves relevant chunks for a user query
5. Generates a grounded answer using Gemini as the LLM

---
## Step 1 - Install Dependencies

In [ ]:
!pip install -q llama-index llama-index-llms-gemini llama-index-embeddings-gemini google-generativeai pypdf

## Step 2 - Set Up Your Gemini API Key
Get a free API key at https://aistudio.google.com/app/apikey

In [ ]:
import os
from google.colab import userdata

#Option A: store your key in Colab Secrets (recommended)
#Go to the key icon on the left sidebar -> add a secret named GOOGLE_API_KEY
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception:
    #Option B: paste it directly (less secure but works)
    GOOGLE_API_KEY = "YOUR_API_KEY_HERE"  # <-- replace this

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("API key set successfully.")

##Step 3 - Upload the PDF
Upload `sample-sdf-document__Project_5_.pdf` using the file picker below.

In [ ]:
from google.colab import files

uploaded = files.upload()  #upload your PDF here
pdf_filename = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_filename}")

##Step 4-Parse the PDF into Documents

Design Choice-PDF Parsing: 
We use LlamaIndex's `SimpleDirectoryReader` with the built-in `pypdf` parser. This handles text-based PDFs cleanly and preserves page level metadata, which is important for tracing answers back to specific pages of the Cytiva document.

In [ ]:
from llama_index.core import SimpleDirectoryReader
import shutil, os

#Move PDF into its own folder so SimpleDirectoryReader can load it
os.makedirs("data", exist_ok=True)
shutil.copy(pdf_filename, f"data/{pdf_filename}")

documents = SimpleDirectoryReader("data").load_data()

print(f"Loaded {len(documents)} document chunk(s) from the PDF.")
print(f"\n--- Preview of first document (first 500 chars) ---")
print(documents[0].text[:500])

##Step 5-Configure Gemini Embedding Model & LLM

Design Choice - Embedding Model: `models/embedding-001` (Gemini)  
- Free tier, no billing required  
- 768-dimensional vectors - good balance of quality and speed  
- Designed to work natively with the Gemini ecosystem  

Design Choice - LLM: `models/gemini-1.5-flash`  
- Fast inference, generous free quota  
- Strong at reading comprehension and table-based Q&A

In [ ]:
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.llms.gemini import Gemini
from llama_index.core import Settings

#Embedding model
Settings.embed_model = GeminiEmbedding(model_name="models/embedding-001")

#LLM
Settings.llm = Gemini(model="models/gemini-1.5-flash")

print("Gemini embedding model and LLM configured.")

##Step 6 - Chunk the Documents

Design Choice - Chunking Strategy: `SentenceSplitter` with `chunk_size=512` and `chunk_overlap=50`  

Why these settings:
- 512 tokens per chunk - large enough to capture full paragraphs and table rows together, but small enough to keep retrieval precise. The Cytiva document has a mix of paragraph text and tabular data; 512 tokens keeps related content (e.g., a product description + its part number + its operating temperature) within the same chunk.
- 50-token overlap- prevents information loss at chunk boundaries. If a sentence about storage conditions gets split, the overlap ensures the full context appears in at least one chunk.
- SentenceSplitter specifically splits on sentence boundaries rather than mid-sentence, which preserves meaning better than a naive character-level split.

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Created {len(nodes)} chunks from the document.\n")
for i, node in enumerate(nodes):
    print(f"--- Chunk {i+1} ({len(node.text)} chars) ---")
    print(node.text[:300])
    print("...\n")

## Step 7 - Build the Vector Index

Design Choice - Retrieval Method:Vector-based retrieval (`VectorStoreIndex`)  

Why vector retrieval over keyword or hybrid:
- Semantic matching - The queries ask about concepts like "quality control" and "storage conditions." The exact words may not appear verbatim in the document (e.g., the document says "Product Release Criteria" not "quality control"). Vector embeddings capture meaning, so semantically similar terms still match.
- Small corpus - With only a few pages, a keyword index (BM25) would struggle with vocabulary mismatch. Vector search handles this gracefully.
- Simplicity - A hybrid approach adds complexity without much benefit for a 3-page document. Pure vector retrieval is sufficient here.

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes)

print("Vector index built successfully.")
print(f"Index contains {len(nodes)} embedded chunks.")

##Step 8 - Create the Query Engine

We set `similarity_top_k=3` so the engine retrieves the 3 most relevant chunks for each query. This gives Gemini enough context to answer accurately without overwhelming it with irrelevant text.

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=3)

print("Query engine ready.")

---
##Step 9 - Query the RAG Pipeline

###Prompt 1: Quality Control Test Methods

In [ ]:
prompt_1 = "What test methods were used for quality control?"

response_1 = query_engine.query(prompt_1)

print("=" * 70)
print(f"PROMPT: {prompt_1}")
print("=" * 70)
print(f"\nRESPONSE:\n{response_1}")
print("\n" + "-" * 70)
print("SOURCE NODES RETRIEVED:")
for i, node in enumerate(response_1.source_nodes):
    print(f"\n  Chunk {i+1} (score: {node.score:.4f}):")
    print(f"  {node.text[:200]}...")

###Prompt 2: Storage Conditions

In [ ]:
prompt_2 = "What are the storage conditions specified in the certificate?"

response_2 = query_engine.query(prompt_2)

print("=" * 70)
print(f"PROMPT: {prompt_2}")
print("=" * 70)
print(f"\nRESPONSE:\n{response_2}")
print("\n" + "-" * 70)
print("SOURCE NODES RETRIEVED:")
for i, node in enumerate(response_2.source_nodes):
    print(f"\n  Chunk {i+1} (score: {node.score:.4f}):")
    print(f"  {node.text[:200]}...")

---
##Summary of Design Choices

| Component | Choice | Rationale |
|-----------|--------|-----------|
| Embedding Model| `models/embedding-001` (Gemini) | Free, 768-dim vectors, native Gemini integration |
| Chunking | `SentenceSplitter`, 512 tokens, 50 overlap | Preserves sentence boundaries; keeps tables + context together |
| Retrieval| Vector search (`VectorStoreIndex`), top-3 | Semantic matching handles vocabulary mismatch in a small corpus |
| LLM | `gemini-1.5-flash` | Fast, accurate, strong on structured document Q&A |